In [1]:
# Notebook 3: ResNet101 & EfficientNetB2 (CLASSIFICATION MODELS ONLY)

# Import essential libraries
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import zipfile
import shutil
import cv2
import pickle
import json
from tqdm.notebook import tqdm
from google.colab import drive
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
from sklearn.decomposition import PCA
import gc
import psutil
import time


# TensorFlow and Keras imports
from tensorflow.keras.applications import ResNet101, EfficientNetB2
from tensorflow.keras.layers import (
    Input, Dense, Dropout, LSTM, Flatten, GlobalAveragePooling2D,
    Add, Reshape, Conv1D, MaxPooling1D, BatchNormalization, Activation
)
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau



# Create directories for results
os.makedirs('notebook5/notebook3Results', exist_ok=True)

# Set seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Configuration parameters
CONFIG = {
    'image_size': (224, 224),        # Standard size for ResNet101 and EfficientNetB2
    'batch_size': 128,               # Larger batch size for A100 GPU
    'angles': ['000', '018', '036', '054', '072', '090', '108', '126', '144', '162', '180'],  # All angles
    'scenarios': ['nm', 'bg', 'cl'],  # Normal, carrying bag, wearing coat
    'sample_ratio': 1,             # Use full dataset since extraction already completed
    'epochs': 32,                    # More epochs with early stopping
    'results_dir': 'notebook5/notebook3Results',
    'early_stopping_patience': 10,   # Increased patience
    'max_samples': None,
    'model_seed': 42                 # For reproducibility
}

# Class mapping for easy reference
CLASS_MAPPING = {
    'nm': 0,  # normal walking
    'bg': 1,  # carrying bag
    'cl': 2   # wearing coat
}

# ONLY ABSOLUTELY NECESSARY FUNCTIONS

def mount_drive():
    """Mount Google Drive to access dataset"""
    drive.mount('/content/drive')

def extract_dataset(zip_path, extract_to):
    """Extract dataset from zip file"""
    os.makedirs(extract_to, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        for file_info in tqdm(zip_ref.infolist(), desc="Extracting files"):
            if not file_info.filename.startswith('__MACOSX'):
                zip_ref.extract(file_info, extract_to)

    macosx_path = os.path.join(extract_to, '__MACOSX')
    if os.path.exists(macosx_path):
        shutil.rmtree(macosx_path)

    return extract_to

def is_valid_image(image_path):
    """Check if an image file is valid"""
    try:
        img = cv2.imread(image_path)
        if img is None or img.shape[2] != 3:
            return False
        return True
    except:
        return False

def get_walking_style_from_path(path):
    """Extract walking style from file path"""
    path_parts = path.split(os.sep)
    for part in path_parts:
        if part.startswith('nm-') or part == 'nm':
            return 'nm'
        elif part.startswith('bg-') or part == 'bg':
            return 'bg'
        elif part.startswith('cl-') or part == 'cl':
            return 'cl'
    return None

def get_angle_from_path(path):
    """Extract viewing angle from file path"""
    path_parts = path.split(os.sep)
    for part in path_parts:
        if part in CONFIG['angles']:
            return part
    return None

def build_dataset_index(base_path, sample_ratio=1.0):
    """Build an index of all valid image files"""
    dataset_index = []
    styles_count = {'nm': 0, 'bg': 0, 'cl': 0}
    angles_count = {angle: 0 for angle in CONFIG['angles']}

    # For visualization in Notebook 5: track detailed counts
    detailed_counts = {
        f"{style}_{angle}": 0
        for style in CONFIG['scenarios']
        for angle in CONFIG['angles']
    }

    for root, _, files in tqdm(list(os.walk(base_path)), desc="Scanning folders"):
        image_files = [f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        if sample_ratio < 1.0 and len(image_files) > 0:
            num_samples = max(1, int(len(image_files) * sample_ratio))
            image_files = np.random.choice(image_files, num_samples, replace=False)

        for file in image_files:
            file_path = os.path.join(root, file)
            walking_style = get_walking_style_from_path(file_path)
            angle = get_angle_from_path(file_path)
            if walking_style and angle and is_valid_image(file_path):
                styles_count[walking_style] += 1
                angles_count[angle] += 1
                # Update detailed counts for Notebook 5
                detailed_counts[f"{walking_style}_{angle}"] += 1

                dataset_index.append({
                    'path': file_path,
                    'style': walking_style,
                    'angle': angle,
                    'style_id': CLASS_MAPPING[walking_style]
                })

    df = pd.DataFrame(dataset_index)

    # Save distribution data for Notebook 5
    distribution_data = {
        'styles_count': styles_count,
        'angles_count': angles_count,
        'detailed_counts': detailed_counts,  # Added for Notebook 5
        'scenarios': CONFIG['scenarios'],
        'angles': CONFIG['angles']
    }
    with open(os.path.join(CONFIG['results_dir'], 'distribution_data.pkl'), 'wb') as f:
        pickle.dump(distribution_data, f)

    return df

def load_dataset_from_drive():
    """Load dataset from Google Drive"""
    mount_drive()
    zip_path = '/content/drive/MyDrive/CASIA B dataset.zip'
    extract_to = '/content/dataset'
    dataset_path = extract_dataset(zip_path, extract_to)
    dataset_df = build_dataset_index(dataset_path, sample_ratio=CONFIG['sample_ratio'])
    return dataset_df, dataset_path

def preprocess_image(image_path, target_size=CONFIG['image_size']):
    """Load and preprocess an image"""
    try:
        img = cv2.imread(image_path)
        img = cv2.resize(img, target_size)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = img.astype(np.float32) / 255.0
        return img
    except:
        return np.zeros((*target_size, 3), dtype=np.float32)

def build_feature_extractors():
    """Build ResNet101 and EfficientNetB2 feature extractor models"""
    models = {}

    # ResNet101
    resnet_base = ResNet101(
        weights='imagenet',
        include_top=False,
        input_shape=(*CONFIG['image_size'], 3)
    )
    for layer in resnet_base.layers[:-30]:  # Unfreeze more layers for better feature extraction
        layer.trainable = False
    models['resnet'] = Model(
        inputs=resnet_base.input,
        outputs=GlobalAveragePooling2D()(resnet_base.output),
        name='resnet_features'
    )

    # EfficientNetB2
    efficientnet_base = EfficientNetB2(
        weights='imagenet',
        include_top=False,
        input_shape=(*CONFIG['image_size'], 3)
    )
    for layer in efficientnet_base.layers[:-30]:  # Unfreeze more layers for better feature extraction
        layer.trainable = False
    models['efficientnet'] = Model(
        inputs=efficientnet_base.input,
        outputs=GlobalAveragePooling2D()(efficientnet_base.output),
        name='efficientnet_features'
    )

    return models

def extract_batch_features(model, image_paths):
    """Extract features for a batch of images"""
    batch_size = len(image_paths)
    batch_images = np.zeros((batch_size, *CONFIG['image_size'], 3), dtype=np.float32)
    for i, path in enumerate(image_paths):
        batch_images[i] = preprocess_image(path)
    features = model.predict(batch_images, verbose=0)
    return features

def process_and_extract_features(models, dataset_df, max_samples=None):
    """Process the dataset and extract features"""
    if max_samples is not None and max_samples < len(dataset_df):
        dataset_df = dataset_df.sample(max_samples)

    total_samples = len(dataset_df)
    features = {model_name: [] for model_name in models.keys()}
    labels = []
    paths = []
    angles = []

    # For visualization in Notebook 5: track processing times
    processing_times = {
        'model': {model_name: [] for model_name in models.keys()},
        'angle': {angle: {model_name: [] for model_name in models.keys()} for angle in CONFIG['angles']}
    }

    batch_size = CONFIG['batch_size']
    for start_idx in tqdm(range(0, total_samples, batch_size), desc="Processing batches"):
        end_idx = min(start_idx + batch_size, total_samples)
        batch_df = dataset_df.iloc[start_idx:end_idx]

        batch_paths = batch_df['path'].values
        batch_labels = batch_df['style_id'].values
        batch_angles = batch_df['angle'].values

        for model_name, model in models.items():
            start_time = time.time()
            batch_features = extract_batch_features(model, batch_paths)
            end_time = time.time()

            # Track processing time for each model
            processing_time = end_time - start_time
            processing_times['model'][model_name].append(processing_time)

            # Track processing time by angle
            for angle in np.unique(batch_angles):
                angle_indices = np.where(batch_angles == angle)[0]
                if len(angle_indices) > 0:
                    avg_time_per_sample = processing_time / len(batch_paths)
                    angle_time = avg_time_per_sample * len(angle_indices)
                    processing_times['angle'][angle][model_name].append(angle_time)

            features[model_name].extend(batch_features)

        labels.extend(batch_labels)
        paths.extend(batch_paths)
        angles.extend(batch_angles)
        gc.collect()

    # Convert lists to numpy arrays
    for model_name in models.keys():
        features[model_name] = np.array(features[model_name])
    labels = np.array(labels)
    angles = np.array(angles)

    # Convert labels to one-hot encoding
    labels_onehot = tf.keras.utils.to_categorical(labels, num_classes=len(CLASS_MAPPING))

    # Compute average processing times
    avg_times = {
        'model': {model_name: np.mean(times) for model_name, times in processing_times['model'].items()},
        'angle': {angle: {model_name: np.mean(times) if times else 0
                        for model_name, times in model_times.items()}
                for angle, model_times in processing_times['angle'].items()}
    }

    # Save features for Notebook 5
    features_data = {
        'features': features,
        'labels': labels_onehot,
        'raw_labels': labels,
        'paths': paths,
        'angles': angles,
        'processing_times': processing_times,  # Raw processing times
        'avg_processing_times': avg_times      # Average processing times
    }
    with open(os.path.join(CONFIG['results_dir'], 'features_data.pkl'), 'wb') as f:
        pickle.dump(features_data, f)

    return features, labels_onehot, paths, angles

def apply_pca_to_features(features, n_components=0.95):
    """Apply PCA to reduce feature dimensionality"""
    pca = PCA(n_components=n_components)
    reduced_features = pca.fit_transform(features)

    # Calculate cumulative variance
    cumulative_variance = np.cumsum(pca.explained_variance_ratio_)

    # Create a safe function to find components for a threshold
    def find_components_for_threshold(threshold):
        indices = np.where(cumulative_variance >= threshold)[0]
        if len(indices) > 0:
            return int(indices[0]) + 1
        else:
            # If threshold is never reached, return the total number of components
            return len(cumulative_variance)

    # Save PCA data for Notebook 5
    pca_data = {
        'explained_variance_ratio': pca.explained_variance_ratio_,
        'explained_variance': pca.explained_variance_,
        'n_components': pca.n_components_,
        'cumulative_variance': cumulative_variance,
        # Add components for specific variance thresholds for visualization
        'variance_thresholds': {
            '50%': find_components_for_threshold(0.5),
            '75%': find_components_for_threshold(0.75),
            '90%': find_components_for_threshold(0.9),
            '95%': find_components_for_threshold(0.95),
            '99%': find_components_for_threshold(0.99)
        }
    }
    with open(os.path.join(CONFIG['results_dir'], 'pca_data.pkl'), 'wb') as f:
        pickle.dump(pca_data, f)

    # Save the PCA model
    with open(os.path.join(CONFIG['results_dir'], 'pca_model.pkl'), 'wb') as f:
        pickle.dump(pca, f)

    return reduced_features, pca

def concatenate_features(feature_dict, apply_pca=True):
    """Concatenate features from multiple models"""
    concatenated = np.concatenate([feature_dict['resnet'], feature_dict['efficientnet']], axis=1)

    if apply_pca:
        # Use more components to retain more information
        reduced_features, pca_model = apply_pca_to_features(concatenated, n_components=768)
        with open(os.path.join(CONFIG['results_dir'], 'concatenated_features.pkl'), 'wb') as f:
            pickle.dump({'raw': concatenated, 'reduced': reduced_features}, f)
        return reduced_features, pca_model

    with open(os.path.join(CONFIG['results_dir'], 'concatenated_features.pkl'), 'wb') as f:
        pickle.dump({'raw': concatenated}, f)
    return concatenated, None

# Enhanced Classification Models

# Enhanced CNN Classifier for better performance
# Enhanced CNN Classifier with proper shape management for residual connections
# Enhanced CNN Classifier with proper shape management for residual connections
def build_enhanced_cnn_classifier(input_shape, num_classes=len(CLASS_MAPPING)):
    """Build an enhanced CNN classifier with deeper architecture and residual connections"""
    inputs = Input(shape=input_shape)

    # First block
    x = Dense(2048, activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    # Residual block 1
    res1 = Dense(1024, activation=None)(x)  # Projection to match dimensions
    x = Dense(1024, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    x = Dense(1024, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Add()([res1, x])  # Now shapes match
    # Use Keras Activation layer instead of tf.nn.relu
    x = Activation('relu')(x)

    # Residual block 2
    res2 = Dense(512, activation=None)(x)  # Projection to match dimensions
    x = Dense(512, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.25)(x)
    x = Dense(512, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Add()([res2, x])  # Now shapes match
    # Use Keras Activation layer instead of tf.nn.relu
    x = Activation('relu')(x)

    # Final layers
    x = Dense(256, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)

    outputs = Dense(num_classes, activation='softmax')(x)
    model = Model(inputs=inputs, outputs=outputs, name="enhanced_cnn_classifier")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0002),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Enhanced LSTM Classifier
def build_enhanced_lstm_classifier(input_shape, num_classes=len(CLASS_MAPPING)):
    """Build an enhanced LSTM classifier with more capacity"""
    model = Sequential(name="enhanced_lstm_classifier")
    model.add(Reshape((1, input_shape[0]), input_shape=input_shape))
    model.add(LSTM(512, return_sequences=True))
    model.add(BatchNormalization())
    model.add(Dropout(0.4))
    model.add(LSTM(256, return_sequences=False))
    model.add(BatchNormalization())
    model.add(Dropout(0.4))
    model.add(Dense(256, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.3))
    model.add(Dense(128, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.2))
    model.add(Dense(num_classes, activation='softmax'))
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# BiLSTM Classifier
def build_bilstm_classifier(input_shape, num_classes=len(CLASS_MAPPING)):
    """Build a Bidirectional LSTM classifier"""
    model = Sequential(name="bilstm_classifier")
    model.add(Reshape((1, input_shape[0]), input_shape=input_shape))
    model.add(tf.keras.layers.Bidirectional(LSTM(512, return_sequences=True)))
    model.add(BatchNormalization())
    model.add(Dropout(0.4))
    model.add(tf.keras.layers.Bidirectional(LSTM(256, return_sequences=False)))
    model.add(BatchNormalization())
    model.add(Dropout(0.4))
    model.add(Dense(256, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.3))
    model.add(Dense(num_classes, activation='softmax'))
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Enhanced 3D CNN Classifier
# Enhanced proper 3D CNN Classifier
def build_enhanced_3dcnn_classifier(input_shape, num_classes=len(CLASS_MAPPING)):
    """Build a proper 3D CNN classifier"""
    inputs = Input(shape=input_shape)

    # Reshape to add spatial dimensions for 3D convolution
    # Convert 1D feature vector to a volume with spatial dimensions
    x = Reshape((input_shape[0] // 16, 4, 4, 1))(inputs)

    # Apply true 3D convolutions
    x = tf.keras.layers.Conv3D(64, kernel_size=(3, 3, 3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling3D(pool_size=(2, 2, 2), padding='same')(x)
    x = Dropout(0.3)(x)

    x = tf.keras.layers.Conv3D(128, kernel_size=(3, 3, 3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling3D(pool_size=(2, 2, 2), padding='same')(x)
    x = Dropout(0.3)(x)

    # Flatten and pass through dense layers
    x = Flatten()(x)

    x = Dense(512, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    x = Dense(256, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.25)(x)

    outputs = Dense(num_classes, activation='softmax')(x)
    model = Model(inputs=inputs, outputs=outputs, name="enhanced_3dcnn_classifier")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0002),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

  # CNN-LSTM Classifier
def build_cnn_lstm_classifier(input_shape, num_classes=len(CLASS_MAPPING)):
    """Build a hybrid CNN-LSTM classifier"""
    # Reshape input for CNN part
    inputs = Input(shape=input_shape)
    reshaped = Reshape((input_shape[0], 1))(inputs)

    # CNN part
    x = Conv1D(128, kernel_size=5, activation='relu', padding='same')(reshaped)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)

    x = Conv1D(256, kernel_size=3, activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)

    # LSTM part
    x = LSTM(384, return_sequences=False)(x)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)

    # Dense layers
    x = Dense(256, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    outputs = Dense(num_classes, activation='softmax')(x)
    model = Model(inputs=inputs, outputs=outputs, name="cnn_lstm_classifier")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0003),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# CNN-BiLSTM Classifier
def build_cnn_bilstm_classifier(input_shape, num_classes=len(CLASS_MAPPING)):
    """Build a hybrid CNN-BiLSTM classifier"""
    # Reshape input for CNN part
    inputs = Input(shape=input_shape)
    reshaped = Reshape((input_shape[0], 1))(inputs)

    # CNN part
    x = Conv1D(128, kernel_size=5, activation='relu', padding='same')(reshaped)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)

    x = Conv1D(256, kernel_size=3, activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)

    # BiLSTM part
    x = tf.keras.layers.Bidirectional(LSTM(384, return_sequences=False))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)

    # Dense layers
    x = Dense(256, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    outputs = Dense(num_classes, activation='softmax')(x)
    model = Model(inputs=inputs, outputs=outputs, name="cnn_bilstm_classifier")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0003),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Train model function with enhanced callbacks
def train_model(model, X_train, y_train, X_val, y_val, epochs=CONFIG['epochs'],
                batch_size=CONFIG['batch_size'], model_name="model"):
    """Train a model with enhanced callbacks"""
    # Use class weights to handle imbalance
    class_counts = np.sum(y_train, axis=0)
    total = np.sum(class_counts)
    class_weights = {i: total / (len(class_counts) * count) for i, count in enumerate(class_counts)}

    # Set patience based on model type
    patience = CONFIG['early_stopping_patience']

    # Enhanced callbacks
    callbacks = [
        EarlyStopping(
            monitor='val_accuracy',
            patience=patience,
            restore_best_weights=True,
            verbose=1,
            min_delta=0.001  # Require more improvement for early stopping
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.3,          # More aggressive reduction
            patience=patience//2,
            min_lr=1e-6,
            verbose=1
        ),
        ModelCheckpoint(
            os.path.join(CONFIG['results_dir'], f"{model_name}_best.h5"),
            monitor='val_accuracy',
            save_best_only=True,
            verbose=1
        )
    ]

    # Time tracking for training
    start_time = time.time()

    # Train model
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        batch_size=batch_size,
        epochs=epochs,
        callbacks=callbacks,
        class_weight=class_weights,
        verbose=1
    )

    # Calculate training time
    training_time = time.time() - start_time

    # Save the model and history
    model.save(os.path.join(CONFIG['results_dir'], f"{model_name}_final.h5"))

    # Enhanced history data for visualization
    history_data = {
        'history': history.history,
        'training_time': training_time,
        'epochs_completed': len(history.history['loss']),
        'best_val_accuracy': max(history.history['val_accuracy']),
        'best_val_loss': min(history.history['val_loss']),
        'final_train_accuracy': history.history['accuracy'][-1],
        'final_train_loss': history.history['loss'][-1]
    }

    with open(os.path.join(CONFIG['results_dir'], f"{model_name}_history.pkl"), 'wb') as f:
        pickle.dump(history_data, f)

    return history, model, training_time

def evaluate_model(model, X_test, y_test, angles_test=None, model_name="model"):
    """Evaluate model and save results with handling for missing classes"""
    # Predict on test set
    y_pred_proba = model.predict(X_test)
    y_pred = np.argmax(y_pred_proba, axis=1)
    y_true = np.argmax(y_test, axis=1)

    # Calculate accuracy and confusion matrix
    accuracy = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)

    # Calculate precision, recall, and F1 score with zero_division=0
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', zero_division=0
    )

    # Store all metrics
    metrics = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'confusion_matrix': cm.tolist()
    }

    # Get unique classes in the data
    unique_classes = sorted(np.unique(np.concatenate([y_true, y_pred])))
    class_names = [list(CLASS_MAPPING.keys())[i] for i in unique_classes]

    # If we have fewer classes than expected, log this
    if len(unique_classes) < len(CLASS_MAPPING):
        print(f"Warning: Only detected {len(unique_classes)} classes in data/predictions: {class_names}")
        print(f"Expected {len(CLASS_MAPPING)} classes: {list(CLASS_MAPPING.keys())}")

    try:
        # Try with explicit labels parameter
        report = classification_report(
            y_true, y_pred,
            labels=unique_classes,
            target_names=class_names,
            output_dict=True,
            zero_division=0
        )
    except Exception as e:
        print(f"Warning: Could not generate classification report: {e}")
        # Fallback to a basic report as dictionary
        report = {
            'accuracy': accuracy,
            'weighted_precision': precision,
            'weighted_recall': recall,
            'weighted_f1': f1
        }

    # Combine metrics and report
    all_metrics = {**metrics, 'classification_report': report}

    with open(os.path.join(CONFIG['results_dir'], f"{model_name}_metrics.json"), 'w') as f:
        json.dump(all_metrics, f)

    # Save confusion matrix
    np.save(os.path.join(CONFIG['results_dir'], f"{model_name}_confusion_matrix.npy"), cm)

    # Calculate and save metrics by angle if angles are provided
    if angles_test is not None:
        angle_metrics = {}
        for angle in np.unique(angles_test):
            angle_idx = np.where(angles_test == angle)[0]

            # Per-angle predictions and ground truth
            angle_y_true = y_true[angle_idx]
            angle_y_pred = y_pred[angle_idx]

            # Skip if no samples for this angle
            if len(angle_y_true) == 0:
                continue

            # Calculate metrics
            angle_acc = accuracy_score(angle_y_true, angle_y_pred)

            try:
                angle_prec, angle_recall, angle_f1, _ = precision_recall_fscore_support(
                    angle_y_true, angle_y_pred, average='weighted', zero_division=0
                )
            except Exception:
                # Fallback if metrics can't be calculated
                angle_prec, angle_recall, angle_f1 = 0.0, 0.0, 0.0

            # Save confusion matrix for each angle (safely)
            try:
                angle_cm = confusion_matrix(angle_y_true, angle_y_pred)
                np.save(os.path.join(CONFIG['results_dir'], f"{model_name}_confusion_matrix_angle_{angle}.npy"), angle_cm)
            except Exception:
                angle_cm = [[0]]

            # Store all per-angle metrics
            angle_metrics[angle] = {
                'accuracy': float(angle_acc),
                'precision': float(angle_prec),
                'recall': float(angle_recall),
                'f1': float(angle_f1),
                'confusion_matrix': angle_cm.tolist() if hasattr(angle_cm, 'tolist') else angle_cm,
                'samples': int(len(angle_y_true))
            }

            # Get per-class metrics for this angle (safely)
            try:
                # Get unique classes in this angle's data
                angle_unique_classes = sorted(np.unique(np.concatenate([angle_y_true, angle_y_pred])))
                angle_class_names = [list(CLASS_MAPPING.keys())[i] for i in angle_unique_classes]

                angle_class_report = classification_report(
                    angle_y_true, angle_y_pred,
                    labels=angle_unique_classes,
                    target_names=angle_class_names,
                    output_dict=True,
                    zero_division=0
                )
                angle_metrics[angle]['classification_report'] = angle_class_report
            except Exception as e:
                print(f"Warning: Could not generate classification report for angle {angle}: {e}")
                # Just include basic metrics
                angle_metrics[angle]['classification_report'] = {
                    'accuracy': float(angle_acc),
                    'weighted_precision': float(angle_prec),
                    'weighted_recall': float(angle_recall),
                    'weighted_f1': float(angle_f1)
                }

        # Save angle metrics
        with open(os.path.join(CONFIG['results_dir'], f"{model_name}_angle_metrics.json"), 'w') as f:
            json.dump(angle_metrics, f)

    print(f"Model: {model_name}, Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")
    print(f"Confusion Matrix:\n{cm}")

    return accuracy, cm, metrics

def train_all_classification_models(concatenated_features, labels, angles, test_size=0.2, val_size=0.1):
    """Train all classification models and save results"""
    # Split data
    X_train_val, X_test, y_train_val, y_test, angles_train_val, angles_test = train_test_split(
        concatenated_features, labels, angles, test_size=test_size,
        stratify=np.argmax(labels, axis=1), random_state=42
    )

    val_ratio = val_size / (1 - test_size)
    X_train, X_val, y_train, y_val, angles_train, angles_val = train_test_split(
        X_train_val, y_train_val, angles_train_val, test_size=val_ratio,
        stratify=np.argmax(y_train_val, axis=1), random_state=42
    )

    print(f"Data split: Train={X_train.shape[0]}, Val={X_val.shape[0]}, Test={X_test.shape[0]}")

    # Save split data for reproducibility
    split_data = {
        'X_train_shape': X_train.shape,
        'X_val_shape': X_val.shape,
        'X_test_shape': X_test.shape,
        'angles_train': angles_train,
        'angles_val': angles_val,
        'angles_test': angles_test,
        'class_distribution_train': np.sum(y_train, axis=0).tolist(),
        'class_distribution_val': np.sum(y_val, axis=0).tolist(),
        'class_distribution_test': np.sum(y_test, axis=0).tolist()
    }
    with open(os.path.join(CONFIG['results_dir'], 'data_split_info.pkl'), 'wb') as f:
        pickle.dump(split_data, f)

    # Enhanced models to train with specific seeds
    models_to_train = [
        ('Enhanced_CNN', build_enhanced_cnn_classifier(X_train.shape[1:]), 123),
        ('Enhanced_LSTM', build_enhanced_lstm_classifier(X_train.shape[1:]), 456),
        ('BiLSTM', build_bilstm_classifier(X_train.shape[1:]), 789),
        ('Enhanced_3DCNN', build_enhanced_3dcnn_classifier(X_train.shape[1:]), 101),
        ('CNN_LSTM', build_cnn_lstm_classifier(X_train.shape[1:]), 202),
        ('CNN_BiLSTM', build_cnn_bilstm_classifier(X_train.shape[1:]), 303)
    ]

    # Train each model
    results = {}
    for model_name, model, seed in models_to_train:
        print(f"\n{'='*50}")
        print(f"Training {model_name} model with seed {seed}...")
        print(f"{'='*50}")

        tf.random.set_seed(seed)
        np.random.seed(seed)

        history, trained_model, training_time = train_model(
            model, X_train, y_train, X_val, y_val,
            epochs=CONFIG['epochs'], batch_size=CONFIG['batch_size'],
            model_name=model_name
        )

        print(f"\nEvaluating {model_name} model...")
        accuracy, cm, metrics = evaluate_model(
            trained_model, X_test, y_test, angles_test=angles_test, model_name=model_name
        )

        results[model_name] = {
            'model': trained_model,
            'history': history,
            'accuracy': accuracy,
            'confusion_matrix': cm,
            'metrics': metrics,
            'training_time': training_time
        }

        # Save important data separately
        with open(os.path.join(CONFIG['results_dir'], f"{model_name}_results.pkl"), 'wb') as f:
            pickle.dump({
                'accuracy': accuracy,
                'confusion_matrix': cm,
                'history': history.history,
                'metrics': metrics,
                'training_time': training_time
            }, f)

    # Create and save performance table with all metrics
    performance_metrics = []
    for name, info in results.items():
        metrics = info['metrics']
        performance_metrics.append({
            'Model': name,
            'Accuracy': metrics['accuracy'],
            'Precision': metrics['precision'],
            'Recall': metrics['recall'],
            'F1': metrics['f1'],
            'Training_Time': info['training_time']
        })

    performance_df = pd.DataFrame(performance_metrics).sort_values('Accuracy', ascending=False)
    performance_df.to_csv(os.path.join(CONFIG['results_dir'], "model_performance.csv"), index=False)

    print("\nModel Performance Ranking:")
    print(performance_df.to_string(index=False))

    # Save combined results
    with open(os.path.join(CONFIG['results_dir'], "all_classification_results.pkl"), 'wb') as f:
        pickle.dump(results, f)

    return results, performance_df

# Main execution function
def run_notebook3():
    """Run Notebook 1 pipeline: ResNet101 & EfficientNetB2 for Classification"""
    print("="*80)
    print("NOTEBOOK 1: RESNET101 & EFFICIENTNETB2 FEATURE EXTRACTION AND CLASSIFICATION MODELS")
    print("="*80)

    # Create a file to track the start time
    start_time = time.time()
    with open(os.path.join(CONFIG['results_dir'], 'execution_times.json'), 'w') as f:
        json.dump({'notebook3_start_time': start_time}, f)

    # Load dataset
    print("\nLoading dataset...")
    dataset_df, dataset_path = load_dataset_from_drive()
    dataset_load_time = time.time()

    # Build feature extractors
    print("\nBuilding feature extractors...")
    feature_extractors = build_feature_extractors()
    feature_extractor_build_time = time.time()

    # Extract features
    print("\nExtracting features...")
    features, labels, paths, angles = process_and_extract_features(
        feature_extractors, dataset_df, max_samples=CONFIG['max_samples']
    )
    feature_extraction_time = time.time()

    # Concatenate features with PCA
    print("\nConcatenating features with PCA...")
    concatenated_features, pca_model = concatenate_features(features, apply_pca=True)
    pca_time = time.time()

    # Train classification models
    print("\nTraining classification models...")
    model_results, performance_df = train_all_classification_models(
        concatenated_features, labels, angles
    )
    model_training_time = time.time()

    # Save best model info
    best_model_name = performance_df.iloc[0]['Model']
    best_model_accuracy = performance_df.iloc[0]['Accuracy']

    # Save execution times
    execution_times = {
        'total_time': model_training_time - start_time,
        'dataset_loading_time': dataset_load_time - start_time,
        'feature_extractor_build_time': feature_extractor_build_time - dataset_load_time,
        'feature_extraction_time': feature_extraction_time - feature_extractor_build_time,
        'pca_time': pca_time - feature_extraction_time,
        'model_training_time': model_training_time - pca_time,
        'models': {name: info['training_time'] for name, info in model_results.items()}
    }

    with open(os.path.join(CONFIG['results_dir'], 'execution_times.json'), 'w') as f:
        json.dump(execution_times, f)

    print(f"\nBest model: {best_model_name} with accuracy {best_model_accuracy:.4f}")
    print(f"\nAll results saved to: {CONFIG['results_dir']}")

    # Create README with results summary
    with open(os.path.join(CONFIG['results_dir'], "README.md"), 'w') as f:
        f.write("# Notebook 1: ResNet101 & EfficientNetB2 Classification Results\n\n")
        f.write("## Feature Extraction\n")
        f.write(f"- ResNet101 feature dimension: {features['resnet'].shape[1]}\n")
        f.write(f"- EfficientNetB2 feature dimension: {features['efficientnet'].shape[1]}\n")
        f.write(f"- Concatenated feature dimension: {concatenated_features.shape[1]}\n\n")
        f.write("## Classification Models Performance\n")
        f.write(performance_df.to_markdown(index=False) + "\n\n")
        f.write(f"### Best Model: {best_model_name}\n")
        f.write(f"- Accuracy: {best_model_accuracy:.4f}\n")
        f.write(f"- Total execution time: {execution_times['total_time']:.2f} seconds\n")

    # Create a summary of data collected for Notebook 5
    notebook5_data_summary = {
        'distribution_data': 'Sample distribution by walking style and angle',
        'features_data': 'Extracted features, labels, processing times',
        'pca_data': 'PCA variance ratios, components, thresholds',
        'concatenated_features': 'Raw and reduced features',
        'execution_times': 'Timing of each pipeline component',
        'model_metrics': 'Performance metrics for each model',
        'model_history': 'Training history for each model',
        'confusion_matrices': 'Overall and per-angle confusion matrices',
        'angle_metrics': 'Detailed performance by viewing angle'
    }

    # Save summary for easier reference
    with open(os.path.join(CONFIG['results_dir'], 'data_summary_for_notebook5.json'), 'w') as f:
        json.dump(notebook5_data_summary, f)

    print("\nNOTEBOOK 1 COMPLETED")
    return {
        'feature_extractors': feature_extractors,
        'features': features,
        'concatenated_features': concatenated_features,
        'pca_model': pca_model,
        'labels': labels,
        'angles': angles,
        'paths': paths,
        'models': model_results,
        'performance': performance_df,
        'execution_times': execution_times
    }

# Run the notebook
if __name__ == "__main__":
    run_notebook3()

NOTEBOOK 1: RESNET101 & EFFICIENTNETB2 FEATURE EXTRACTION AND CLASSIFICATION MODELS

Loading dataset...
Mounted at /content/drive



Building feature extractors...
171446536/171446536 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
31790344/31790344 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

Extracting features...



Concatenating features with PCA...

Training classification models...
Data split: Train=396469, Val=56639, Test=113278


/usr/local/lib/python3.11/dist-packages/keras/src/layers/reshaping/reshape.py:39: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



Training Enhanced_CNN model with seed 123...
Epoch 1/32
3098/3098 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6561 - loss: 0.8035
Epoch 1: val_accuracy improved from -inf to 0.83082, saving model to notebook5/notebook3Results/Enhanced_CNN_best.h5


...

3098/3098 ━━━━━━━━━━━━━━━━━━━━ 100s 32ms/step - accuracy: 0.7504 - loss: 0.5959 - val_accuracy: 0.7719 - val_loss: 0.5643 - learning_rate: 3.0000e-04
Epoch 14/32
3097/3098 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.7570 - loss: 0.5769
Epoch 14: val_accuracy did not improve from 0.77189
3098/3098 ━━━━━━━━━━━━━━━━━━━━ 99s 32ms/step - accuracy: 0.7570 - loss: 0.5769 - val_accuracy: 0.6849 - val_loss: 0.7416 - learning_rate: 3.0000e-04
Epoch 15/32
3097/3098 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.7632 - loss: 0.5602
Epoch 15: val_accuracy did not improve from 0.77189
3098/3098 ━━━━━━━━━━━━━━━━━━━━ 99s 32ms/step - accuracy: 0.7632 - loss: 0.5602 - val_accuracy: 0.7348 - val_loss: 0.6349 - learning_rate: 3.0000e-04
Epoch 16/32
3097/3098 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.7713 - loss: 0.5420
Epoch 16: val_accuracy improved from 0.77189 to 0.79761, saving model to notebook5/notebook3Results/CNN_BiLSTM_best.h5


3098/3098 ━━━━━━━━━━━━━━━━━━━━ 99s 32ms/step - accuracy: 0.7713 - loss: 0.5420 - val_accuracy: 0.7976 - val_loss: 0.5146 - learning_rate: 3.0000e-04
Epoch 17/32
3097/3098 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.7773 - loss: 0.5260
Epoch 17: val_accuracy did not improve from 0.79761
3098/3098 ━━━━━━━━━━━━━━━━━━━━ 99s 32ms/step - accuracy: 0.7773 - loss: 0.5260 - val_accuracy: 0.7304 - val_loss: 0.6499 - learning_rate: 3.0000e-04
Epoch 18/32
3097/3098 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.7817 - loss: 0.5089
Epoch 18: val_accuracy did not improve from 0.79761
3098/3098 ━━━━━━━━━━━━━━━━━━━━ 99s 32ms/step - accuracy: 0.7817 - loss: 0.5089 - val_accuracy: 0.7160 - val_loss: 0.6839 - learning_rate: 3.0000e-04
Epoch 19/32
3097/3098 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.7887 - loss: 0.4962
Epoch 19: val_accuracy did not improve from 0.79761
3098/3098 ━━━━━━━━━━━━━━━━━━━━ 99s 32ms/step - accuracy: 0.7887 - loss: 0.4962 - val_accuracy: 0.6828 - val_loss: 0.7386 - lear

3098/3098 ━━━━━━━━━━━━━━━━━━━━ 99s 32ms/step - accuracy: 0.8221 - loss: 0.4056 - val_accuracy: 0.8047 - val_loss: 0.5068 - learning_rate: 9.0000e-05
Epoch 25/32
3098/3098 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.8253 - loss: 0.3975
Epoch 25: val_accuracy did not improve from 0.80469
3098/3098 ━━━━━━━━━━━━━━━━━━━━ 99s 32ms/step - accuracy: 0.8253 - loss: 0.3975 - val_accuracy: 0.8009 - val_loss: 0.5139 - learning_rate: 9.0000e-05
Epoch 26/32
3097/3098 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.8273 - loss: 0.3891
Epoch 26: val_accuracy did not improve from 0.80469
3098/3098 ━━━━━━━━━━━━━━━━━━━━ 99s 32ms/step - accuracy: 0.8273 - loss: 0.3891 - val_accuracy: 0.7924 - val_loss: 0.5311 - learning_rate: 9.0000e-05
Epoch 27/32
3097/3098 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.8290 - loss: 0.3845
Epoch 27: val_accuracy did not improve from 0.80469
3098/3098 ━━━━━━━━━━━━━━━━━━━━ 99s 32ms/step - accuracy: 0.8290 - loss: 0.3845 - val_accuracy: 0.7134 - val_loss: 0.7021 - lear

3098/3098 ━━━━━━━━━━━━━━━━━━━━ 100s 32ms/step - accuracy: 0.8435 - loss: 0.3502 - val_accuracy: 0.8133 - val_loss: 0.5022 - learning_rate: 2.7000e-05
Restoring model weights from the end of the best epoch: 32.



Evaluating CNN_BiLSTM model...
3540/3540 ━━━━━━━━━━━━━━━━━━━━ 27s 8ms/step
Model: CNN_BiLSTM, Accuracy: 0.8123, Precision: 0.8198, Recall: 0.8123, F1: 0.8147
Confusion Matrix:
[[57597  5565  5573]
 [ 3980 16270  1944]
 [ 2728  1469 18152]]

Model Performance Ranking:
         Model  Accuracy  Precision   Recall       F1  Training_Time
  Enhanced_CNN  0.955437   0.955288 0.955437 0.955215     422.808908
        BiLSTM  0.936749   0.937805 0.936749 0.937025    1118.534189
 Enhanced_LSTM  0.931364   0.932507 0.931364 0.931678     986.154506
      CNN_LSTM  0.836888   0.834723 0.836888 0.835347    1820.018384
Enhanced_3DCNN  0.813203   0.830639 0.813203 0.817027     398.503748
    CNN_BiLSTM  0.812329   0.819790 0.812329 0.814676    3187.191604

Best model: Enhanced_CNN with accuracy 0.9554

All results saved to: notebook5/notebook3Results

NOTEBOOK 1 COMPLETED
